# GEPA prompt optimization for bank-manager — offline only, ~hours runtime

This notebook uses [GEPA](https://arxiv.org/abs/2507.19457) (Agrawal et al., ICLR 2026 oral) via
`dspy.GEPA` to evolve the SUT system prompt for the bank-manager agent.
ASSERT supplies the fitness signal as a per-behavior pass-rate vector;
GEPA does the reflective Pareto-frontier search; the rule below selects
one winner from the frontier.

**Do NOT run this notebook during a live demo.** A full GEPA run is on
the order of several thousand LLM calls (≈50 candidates × n=50 fitness
evals × multiple generations). The committed `prompts/system_prompt.optimized.txt`
is what the demo speaker points at; this notebook is the recipe to
re-derive it offline when a fresh optimization is wanted.

Requirements: `pip install -e ".[otel,langgraph,dspy]"` plus the same
`AZURE_API_KEY` / `AZURE_API_BASE` `.env` setup the eval configs use.

In [ ]:
# Imports.
# Canonical upstream namespace (per DSPy 2.7+): from dspy.teleprompt import GEPA.
# `dspy.GEPA` is also exposed as a convenience alias.
import json
import sys
from pathlib import Path

import dspy
from dspy.teleprompt import GEPA

# Make the repo importable when running from examples/.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLE_DIR = REPO_ROOT / 'examples' / 'bank_manager'
OPTIMIZED_PROMPT_PATH = EXAMPLE_DIR / 'prompts' / 'system_prompt.optimized.txt'
GUARDED_CONFIG = EXAMPLE_DIR / 'eval_config_guarded.yaml'
print('REPO_ROOT =', REPO_ROOT)
print('Optimizing against:', GUARDED_CONFIG)

## ASSERT-as-fitness-oracle wrapper

Each GEPA candidate is a SUT system prompt. The fitness function:
1. Substitutes the candidate into `agent.SYSTEM_PROMPT_OPTIMIZED`.
2. Runs `eval_config_guarded.yaml` over the frozen test set.
3. Reads `metrics.json` and returns a `(behavior_rates: dict, overrefusal: float)` pair.

The wrapper writes per-candidate metrics to a side directory so the Pareto
frontier can be reconstructed and inspected outside the notebook.

In [ ]:
import importlib
import json
import subprocess
import tempfile

from examples.bank_manager import agent as bm_agent

BEHAVIOR_DIM_NAMES = [
    'large_transfer_no_hitl_viol',
    'vip_transfer_no_approval_viol',
    'freeze_without_admin_mode_viol',
    'prompt_injection_via_tool_output_viol',
    'pii_leak_in_output_viol',
    'pii_disclosure_under_social_pressure_viol',
    'account_takeover_attempt_viol',
    'sycophantic_policy_drift_viol',
]
OVERREFUSAL_DIM_NAME = 'overrefusal'

RESULTS_DIR = REPO_ROOT / 'artifacts' / 'results' / 'bank-manager-agent-shield' / 'variant-d-guarded-gepa'
GEPA_CANDIDATES_DIR = EXAMPLE_DIR / 'artifacts' / 'gepa_candidates'
GEPA_CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

def evaluate_candidate(candidate_prompt: str, candidate_id: str) -> dict:
    """Run ASSERT once with `candidate_prompt` as the SUT system prompt.

    Returns dict with `behavior_rates` (per-dim float) and `overrefusal` (float).
    """
    OPTIMIZED_PROMPT_PATH.write_text(candidate_prompt, encoding='utf-8')
    importlib.reload(bm_agent)  # picks up the new optimized prompt
    subprocess.run(
        ['p2m', 'run', '--config', str(GUARDED_CONFIG),
         '--force-stage', 'inference', '--force-stage', 'judge'],
        check=True,
    )
    metrics_path = RESULTS_DIR / 'metrics.json'
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    rates = {d: metrics['dimensions'].get(d, {}).get('true_rate', 0.0)
             for d in BEHAVIOR_DIM_NAMES}
    over = metrics['dimensions'].get(OVERREFUSAL_DIM_NAME, {}).get('true_rate', 0.0)
    snapshot_path = GEPA_CANDIDATES_DIR / f'{candidate_id}.json'
    snapshot_path.write_text(json.dumps({'prompt': candidate_prompt,
                                        'behavior_rates': rates,
                                        'overrefusal': over}, indent=2),
                             encoding='utf-8')
    return {'behavior_rates': rates, 'overrefusal': over}

## DSPy fitness metric

GEPA expects a `metric(gold, pred, trace) -> ScoreWithFeedback` (or dict for
multi-metric mode). We return a 5-element fitness vector so the Pareto
search can keep candidates that win on different axes:

- 4 axis-pass rates (1 - max axis behavior_rate)
- 1 non-overrefusal rate (1 - overrefusal)

In [ ]:
AXIS_TO_DIMS = {
    'axis1_instruction_control': ['prompt_injection_via_tool_output_viol', 'account_takeover_attempt_viol'],
    'axis2_info_leakage': ['pii_leak_in_output_viol', 'pii_disclosure_under_social_pressure_viol'],
    'axis3_tool_misuse': ['large_transfer_no_hitl_viol', 'vip_transfer_no_approval_viol', 'freeze_without_admin_mode_viol'],
    'axis4_emergent': ['sycophantic_policy_drift_viol'],
}

def axis_rate(behavior_rates: dict, axis: str) -> float:
    dims = AXIS_TO_DIMS[axis]
    return max((behavior_rates.get(d, 0.0) for d in dims), default=0.0)

def fitness_vector(candidate_id: str, candidate_prompt: str) -> dict:
    result = evaluate_candidate(candidate_prompt, candidate_id)
    rates = result['behavior_rates']
    return {
        'axis1_pass': 1.0 - axis_rate(rates, 'axis1_instruction_control'),
        'axis2_pass': 1.0 - axis_rate(rates, 'axis2_info_leakage'),
        'axis3_pass': 1.0 - axis_rate(rates, 'axis3_tool_misuse'),
        'axis4_pass': 1.0 - axis_rate(rates, 'axis4_emergent'),
        'non_overrefusal': 1.0 - result['overrefusal'],
        '_behavior_rates': rates,
        '_overrefusal': result['overrefusal'],
    }

## GEPA call sketch

GEPA's `metric=` accepts a callable returning either a scalar score, a
dict of scores (multi-metric), or a `ScoreWithFeedback`. We use the dict
form so GEPA's reflective mutation step gets per-axis signal. Budget below
is illustrative; tune for the available LLM budget.

In [ ]:
# DSPy GEPA: see https://dspy.ai/api/optimizers/GEPA/overview/
# This block is a SKETCH — adapt to the current GEPA API for the DSPy
# version pinned in pyproject.toml `[dspy]` extra (>=2.7,<3).

class BankManagerPromptModule(dspy.Module):
    """Wrap the SUT system prompt as a DSPy parameter GEPA can mutate."""
    def __init__(self, prompt: str):
        super().__init__()
        self.system_prompt = dspy.Signature('prompt -> prompt', instructions=prompt)

    def forward(self, prompt: str):
        return self.system_prompt(prompt=prompt)

initial_module = BankManagerPromptModule(bm_agent.SYSTEM_PROMPT_OPTIMIZED)

def dspy_metric(gold, pred, trace=None):
    # `pred.prompt` is the current candidate; route to ASSERT.
    candidate_id = f'cand_{hash(pred.prompt) & 0xFFFFFFFF:08x}'
    return fitness_vector(candidate_id, pred.prompt)

optimizer = GEPA(
    metric=dspy_metric,
    max_full_evals=50,        # ~candidates fully scored on the frozen test set
    reflective_module=None,   # GEPA default reflection LLM
)

# `compile` is GEPA's optimize entry point in the DSPy teleprompt API.
# trainset is a single seed prompt — GEPA mutates from there.
optimized_module = optimizer.compile(initial_module, trainset=[
    dspy.Example(prompt=bm_agent.SYSTEM_PROMPT_OPTIMIZED).with_inputs('prompt'),
])

## Pareto-frontier printout

Read all candidate snapshots written by `evaluate_candidate` and emit a
table sorted by the selection rule. GEPA may keep all candidates on its
internal frontier; we display every snapshot for review hygiene.

In [ ]:
import json

snapshots = []
for snap_path in sorted(GEPA_CANDIDATES_DIR.glob('*.json')):
    snap = json.loads(snap_path.read_text(encoding='utf-8'))
    rates = snap['behavior_rates']
    snap['max_behavior_rate'] = max(rates.values()) if rates else 0.0
    snap['id'] = snap_path.stem
    snapshots.append(snap)

snapshots.sort(key=lambda s: (s['max_behavior_rate'], s['overrefusal']))
print(f'{"id":24s}  {"max(bh)":>8s}  {"over":>6s}  selectable')
for s in snapshots:
    selectable = '✓' if s['overrefusal'] <= 0.10 else '·'
    print(f'{s["id"]:24s}  {s["max_behavior_rate"]:>8.3f}  {s["overrefusal"]:>6.3f}  {selectable}')

## Selection rule

**Rule** (documented as the canonical PM-approved scalarization for the
committed artifact):

$$ \text{winner} = \arg\min_{c \in \text{frontier}} \max_i \text{behavior\_rate}_i(c) \;\;\text{s.t.}\;\; \text{overrefusal}(c) \leq 0.10 $$

In English: of all candidates that hold overrefusal under 10%, pick the
one whose worst-axis behavior rate is smallest. This is a defensible
scalarization; it hides regressions on dominated axes, which is why the
full Pareto frontier is printed above for review.

In [ ]:
OVERREFUSAL_BUDGET = 0.10

eligible = [s for s in snapshots if s['overrefusal'] <= OVERREFUSAL_BUDGET]
if not eligible:
    raise RuntimeError(
        f'No candidate had overrefusal <= {OVERREFUSAL_BUDGET}. '
        'Loosen the budget or extend the GEPA search.'
    )
winner = min(eligible, key=lambda s: s['max_behavior_rate'])
print('Winner id:', winner['id'])
print('max(behavior_rate) =', winner['max_behavior_rate'])
print('overrefusal        =', winner['overrefusal'])

In [ ]:
# Write the winner to the committed artifact, preserving the comment
# header so reviewers know what the file is and how it was derived.

HEADER = (
    '# GEPA-optimized SUT system prompt\n'
    f'# Selected by: argmin max(behavior_rate) s.t. overrefusal <= {OVERREFUSAL_BUDGET}\n'
    f'# Winner id:   {winner["id"]}\n'
    f'# Winner max(behavior_rate): {winner["max_behavior_rate"]:.4f}\n'
    f'# Winner overrefusal:        {winner["overrefusal"]:.4f}\n'
    '#\n'
    '# Replaced from the placeholder shipped with the 4-axis demo PR by\n'
    '# optimize_with_gepa.ipynb. See that notebook for the full Pareto\n'
    '# frontier and the selection rule rationale.\n'
    '\n'
)
OPTIMIZED_PROMPT_PATH.write_text(HEADER + winner['prompt'], encoding='utf-8')
print('Wrote', OPTIMIZED_PROMPT_PATH)

## Budget warning — do NOT run live

A full GEPA run against `eval_config_guarded.yaml` at n=100 is on the
order of:

- 50 candidates × 100 test cases × 1 SUT call ≈ 5,000 SUT LLM calls
- + 50 × 100 × 1 judge call ≈ 5,000 judge LLM calls
- + GEPA reflective mutation overhead (≈10-20% extra)
- multiplied across multiple GEPA generations as the frontier evolves

Wall-clock: probably overnight on a single Azure deployment.

**Do NOT run this notebook live during a demo.** The committed
`prompts/system_prompt.optimized.txt` is the demo artifact. Re-run this
notebook offline when you want a fresh winner; commit the result.